<a href="https://colab.research.google.com/github/VildanaRazumova/thesis-demand-forecasting/blob/main/thesis_notebook_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Demand-Based Dynamic Pricing Models for Travel Markets

---

# The following Project information



**Client:** A leading B2B tour operator in Kazakhstan and Central Asia, cooperating with 5,000+ travel agencies across destinations including Vietnam, Thailand, Egypt, Turkey, UAE and the Maldives

**Problem statement & Importance:**
Tour operators in B2B travel markets must commit to flight and hotel inventory 6-8 months in advance, without knowing the actual demand. Traditional demand estimation methods rely on historical bookings and expert judgment, which becomes insufficient when market conditions shift due to competitor pricing, seasonal search trends, and external events. This uncertainty leads to either unsold seats or missed revenue opportunities. A critical business challenge is not only whether demand can be predicted accurately, but also how early in the booking window a reliable prediction can be made, giving the operator enough time to act.

**Goal:**
This study focuses on the Kazakhstan → Vietnam route as a pilot, as Vietnam is the operator's most popular destination and each route has unique demand patterns, seasonality, and external signals.

The methodology is designed to be scalable and applicable to other destinations in subsequent research. The primary objective is to develop and evaluate machine learning models that predict flight load factor at multiple booking windows (D-90, D-60, D-30, D-7), and to assess whether external market signals improve prediction accuracy beyond internal booking data alone.

---

Load Factor = Seats Sold / Total Seats x 100%

---

**Business Benefits:**
1. Identifies the earliest reliable prediction window
2. Reduces financial risk from unsold charter seats
3. Provides a data-driven foundation for dynamic price decisions
4. Competitive advantage over operators using rule-based pricing only

**Relevant data collected from:**

Historical Internal signals (2024-2026):
-  booking claims: cancellations, lead time, pax, hotel & flight costs
-  flight capacity: seats sold, empty seats, load factor per window

External signals at specific points in the past:
- Google Trends API: search interest per destination
- Google News API: news per country
- Competitor websites: price monitoring
- Public holiday calendars: event & holiday flags

**Approach:**

A time-series approach is used throughout this project. Features are constructed at each booking window using only information available at that point in time, preventing data leakage and ensuring realistic future deployment.

**Core Task: Load Factor Prediction at multiple booking windows**

| Window | Days Before Departure | Business Meaning |
|--------|----------------------|------------------|
| D-90 | 90 days | Early signal: low booking activity |
| D-60 | 60 days | Search trends become informative |
| D-30 | 30 days | Majority of bookings visible |
| D-7  | 7 days  | Near-final occupancy picture |

For each window two experiments are conducted:
- Baseline model: internal features only (Linear Regression)
- Extended model: internal + external signals (Random Forest, XGBoost, LightGBM)

# Project Pipeline

This project will be approached through the following steps:

1. **Importing the Libraries**: loading all required Python packages
   for data analysis, visualisation and machine learning

2. **Importing the Data**: loading claims and flight load history
   datasets from Google Drive

3. **Data Overview**: understanding the structure, shape, date ranges
   and column descriptions of both datasets

4. **Exploratory Data Analysis (EDA)**: analysing demand patterns,
   seasonality, lead time distribution and booking window behaviour
   for Vietnam routes

5. **Data Preprocessing**: filtering for Vietnam, handling data types,
   dropping irrelevant columns, joining claims with flights

6. **Feature Engineering**: constructing internal features (booking
   pace, lead time, cancellation rate) and external signals
   (price ratio, search trends, holidays) at each booking window
   (D-90, D-60, D-30, D-7)

7. **Model Building**: training baseline models (internal features
   only) and extended models (internal + external features) using
   Linear Regression, Random Forest, XGBoost and LightGBM

8. **Model Evaluation**: comparing model performance across booking
   windows using MAPE, MAE, RMSE and R2

9. **Conclusion & Pipeline Summary**: identifying the most reliable
   prediction window and discussing business implications

# 1. Importing the Libraries

In [5]:
# Import Libraries
from google.colab import drive

# Data libraries
import pandas as pd
import numpy as np

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# 2. Import the Data

In [10]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Load Claims

In [22]:
df_claims = pd.read_csv(
    '/content/drive/MyDrive/Thesis_RazumovaV_2026/thesis_data/_dp_claims_features.csv'
)

In [11]:
df_claims.head(3)

,snapshot_date,snapshot_ts_utc,ClaimID,TourID,StatusID,StatusName,ClaimCreatedDT_UTC,raw_event_time_utc,DepartureDT,TourEndDT,...,FlightPartnerID_Fwd,FlightPartner_Fwd,FlightPartnerID_Bck,FlightPartner_Bck,FlightCost_Fwd,FlightCost_Bck,RevisionAmount_Fwd,RevisionAmount_Bck,CurrencyClaimID,CurrencyAlias
0,2026-03-15,2026-03-15 11:24:39.227,11286829,27194,2,Unpaid,2026-03-05 22:10:00.000,2026-03-05 22:10:00.000,2026-05-02 00:00:00.000,2026-05-10 00:00:00.000,...,157575,SCAT (),157575,SCAT (),596.56,736.56,-70.0,0.0,2,USD
1,2026-03-15,2026-03-15 11:24:39.227,11286196,30853,2,Unpaid,2026-03-05 17:44:00.000,2026-03-05 17:44:00.000,2026-06-04 00:00:00.000,2026-06-14 00:00:00.000,...,157575,SCAT (),157575,SCAT (),928.80,1198.80,-90.0,0.0,2,USD
2,2026-03-15,2026-03-15 11:24:39.227,11286197,30853,2,Unpaid,2026-03-05 17:44:00.000,2026-03-05 17:44:00.000,2026-06-04 00:00:00.000,2026-06-14 00:00:00.000,...,157575,SCAT (),157575,SCAT (),619.20,799.20,-90.0,0.0,2,USD


### Load Flights

In [16]:
df_flights = pd.read_csv(
    '/content/drive/MyDrive/Thesis_RazumovaV_2026/thesis_data/_dp_flight_load_history.csv',
    on_bad_lines='skip'
)

In [17]:
df_flights.head(3)

,snapshot_date,snapshot_ts_utc,FreightID,FlightName,BlockDate,AirlinePartnerID,AirlineName,FlightClassID,ClassAlias,DepartureCityID,...,IsOnRequest,Seats_gross,Sold_gross,Empty,Seats_net,Sold_net,BlockRecords,TotalDaysInSale,ResourceCount,last_stamp
0,2026-03-15,2026-03-15 11:31:47.773,10670,VSV5208,2026-03-05,157575,SCAT (),2,Y,838396,...,0,64,64,0,64,64,2,230,1,\u0000\u0000\u0000\u000b0\u0013Y�
1,2026-03-15,2026-03-15 11:31:47.773,13556,VSV5338,2026-03-05,157575,SCAT (),2,Y,378598,...,0,91,91,0,72,72,1,254,1,\u0000\u0000\u0000\u000b7�n�
2,2026-03-15,2026-03-15 11:31:47.773,13557,VSV5339,2026-03-05,157575,SCAT (),2,Y,293967,...,0,91,91,0,83,83,1,254,1,\u0000\u0000\u0000\u000b3���


# Data Overview

In [35]:
print('df_claims columns:')
print(list(df_claims.columns))

df_claims columns:
['snapshot_date', 'snapshot_ts_utc', 'ClaimID', 'TourID', 'StatusID', 'StatusName', 'ClaimCreatedDT_UTC', 'raw_event_time_utc', 'DepartureDT', 'TourEndDT', 'ConfirmedDT_UTC', 'LeadTimeDays', 'TourNights', 'TourNightsCalc', 'HotelNights', 'StateID', 'CountryName', 'TourTypeID', 'TourTypeName', 'Padult', 'Pchild', 'Pinfant', 'PaxSeats', 'PaxHotel', 'HotelID', 'HotelName', 'StarID', 'HotelStars', 'MealID', 'MealName', 'RoomTypeID', 'RoomTypeName', 'AccommodationID', 'AccommodationName', 'HotelRegionID', 'HotelRegion', 'HotelCost', 'HotelCount', 'TotalHotelCost', 'FreightID_Fwd', 'FreightID_Bck', 'DepartureCityID', 'DepartureCityName', 'ArrivalCityID', 'ArrivalCityName', 'ReturnDepartureCityID', 'ReturnDepartureCityName', 'ReturnArrivalCityID', 'ReturnArrivalCityName', 'FlightClassID_Fwd', 'FlightClass_Fwd', 'FlightClassID_Bck', 'FlightClass_Bck', 'FlightPartnerID_Fwd', 'FlightPartner_Fwd', 'FlightPartnerID_Bck', 'FlightPartner_Bck', 'FlightCost_Fwd', 'FlightCost_Bck', '

In [38]:
print('GENERAL OVERVIEW: df_claims')
print(f'Rows: {df_claims.shape[0]:,}')
print(f'Columns: {df_claims.shape[1]}')
print(f'Booking date: {df_claims["ClaimCreatedDT_UTC"].min()[:10]} - {df_claims["ClaimCreatedDT_UTC"].max()[:10]}')
print(f'Departure: {df_claims["DepartureDT"].min()[:10]} - {df_claims["DepartureDT"].max()[:10]}')
print(f'Destination: {df_claims["CountryName"].unique().tolist()}')
print(f'Missing: {df_claims.isnull().sum().sum()}')

GENERAL OVERVIEW: df_claims
Rows: 56,362
Columns: 63
Booking date: 2024-01-02 - 2026-03-05
Departure: 2024-01-11 - 2026-10-08
Destination: ['Vietnam']
Missing: 0


The claims dataset contains **56,362 rows** and **63 columns** (future features), covering
the period from **January 2024 to March 2026**. Each row represents a
single tour package claim for the destination **Vietnam**.

The dataset includes the following key information:
- **Booking behaviour:** claim creation date, departure date, lead time
  in days, tour duration, number of nights
- **Passengers:** number of adults, children, and infants per booking
- **Product details:** hotel name, star rating, meal plan, room type,
  accommodation type, hotel region
- **Pricing:** hotel cost, flight cost (forward and return),
  total hotel cost, revision amounts
- **Flight information:** departure and arrival cities, flight class,
  airline partner, freight IDs

Although the operator works in a B2B model selling packages through
travel agencies each claim represents an end customer (B2C) booking
created by an agent on behalf of their client.


In [37]:
print('df_flights columns:')
print(list(df_flights.columns))

df_flights columns:
['snapshot_date', 'snapshot_ts_utc', 'FreightID', 'FlightName', 'BlockDate', 'AirlinePartnerID', 'AirlineName', 'FlightClassID', 'ClassAlias', 'DepartureCityID', 'CityFrom', 'ArrivalCityID', 'CityTo', 'Country', 'PartnerID', 'BusinessEntity', 'hard_block', 'releasedays', 'first_reserve_utc', 'HasStopSale', 'StopSaleIssueDT_UTC', 'IsOnRequest', 'Seats_gross', 'Sold_gross', 'Empty', 'Seats_net', 'Sold_net', 'BlockRecords', 'TotalDaysInSale', 'ResourceCount', 'last_stamp']


In [32]:
print('GENERAL OVERVIEW: df_flights')
print(f'Rows: {df_flights.shape[0]:,}')
print(f'Columns: {df_flights.shape[1]}')
print(f'BlockDate: {df_flights["BlockDate"].min()[:10]} - {df_flights["BlockDate"].max()[:10]}')
print(f'Missing: {df_flights.isnull().sum().sum()}')

GENERAL OVERVIEW: df_flight
Rows: 3,951
Columns: 31
BlockDate: 2024-01-01 - 2026-03-05
Missing: 0


The flight load history dataset contains **3,951 rows** and **31 columns**,
covering charter flight blocks operated on Kazakhstan -> Vietnam routes, from **January 2024 to March 2026**.

Each row represents a single charter flight block assigned to the
operator, containing capacity and sales information at the time of
the data snapshot.

The dataset includes the following key information:
- **Flight details:** flight name, block date, airline partner,
  departure and arrival cities, flight class
- **Capacity:** gross seats available, net seats available
- **Demand signals:** seats sold (gross and net), empty seats,
  total days in sale
- **Operational flags:** hard block indicator, stop sale flag,
  on-request flag, release days

**Key target variable:**
Load Factor = Sold_gross / Seats_gross × 100%

**Note:** The last_stamp column was excluded due to corrupted
binary data from the export process, it carries no analytical value.

**Missing values:** None detected